# Monte Carlo Methods: Modular Generalized Policy Iteration (GPI)

This notebook provides a comprehensive pedagogical demonstration of Monte Carlo (MC) methods. We break the algorithm into modular components to illustrate the inputs and outputs of each phase and compare different exploration strategies.

## 1. Visual Overview: How MC Works

<svg width="800" height="350" viewBox="0 0 800 350" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { fill: #ffffff; stroke: #343a40; stroke-width: 2; }
    .title { font-family: sans-serif; font-size: 16px; font-weight: bold; fill: #212529; }
    .desc { font-family: sans-serif; font-size: 12px; fill: #495057; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 2; marker-end: url(#arrowhead); }
    .highlight { fill: #f1f8ff; stroke: #007bff; }
    .step-num { font-family: sans-serif; font-size: 20px; font-weight: bold; fill: #dee2e6; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <!-- Step 1 -->
  <rect x="10" y="50" width="180" height="120" class="box highlight" rx="8" />
  <text x="20" y="40" class="step-num">01</text>
  <text x="100" y="80" class="title" text-anchor="middle">Experience</text>
  <text x="100" y="105" class="desc" text-anchor="middle">Pick a RANDOM state & action</text>
  <text x="100" y="120" class="desc" text-anchor="middle">(Exploring Start). Follow the</text>
  <text x="100" y="135" class="desc" text-anchor="middle">current Policy to the Goal.</text>
  <!-- Step 2 -->
  <rect x="210" y="50" width="180" height="120" class="box" rx="8" />
  <text x="220" y="40" class="step-num">02</text>
  <text x="300" y="80" class="title" text-anchor="middle">Backtracking</text>
  <text x="300" y="105" class="desc" text-anchor="middle">The game ends. Look back</text>
  <text x="300" y="120" class="desc" text-anchor="middle">at the path and calculate</text>
  <text x="300" y="135" class="desc" text-anchor="middle">the total Return (G).</text>
  <!-- Step 3 -->
  <rect x="410" y="50" width="180" height="120" class="box" rx="8" />
  <text x="420" y="40" class="step-num">03</text>
  <text x="500" y="80" class="title" text-anchor="middle">Evaluation</text>
  <text x="500" y="105" class="desc" text-anchor="middle">Update scorecard N(s,a).</text>
  <text x="500" y="120" class="desc" text-anchor="middle">Find the NEW average Q(s,a)</text>
  <text x="500" y="135" class="desc" text-anchor="middle">using the observed return G.</text>
  <!-- Step 4 -->
  <rect x="610" y="50" width="180" height="120" class="box" rx="8" />
  <text x="620" y="40" class="step-num">04</text>
  <text x="700" y="80" class="title" text-anchor="middle">Improvement</text>
  <text x="700" y="105" class="desc" text-anchor="middle">Update the map. Pick the</text>
  <text x="700" y="120" class="desc" text-anchor="middle">action with the highest</text>
  <text x="700" y="135" class="desc" text-anchor="middle">historical average (Greedy).</text>
  <!-- Arrows -->
  <path d="M 190 110 L 205 110" class="arrow" />
  <path d="M 390 110 L 405 110" class="arrow" />
  <path d="M 590 110 L 605 110" class="arrow" />
  <path d="M 700 170 L 700 220 L 100 220 L 100 180" class="arrow" />
  <text x="400" y="250" class="title" text-anchor="middle" style="fill:#007bff">The GPI Cycle (Repeat Thousands of Times)</text>
</svg>

## 2. Mathematical Foundations

### The Return ($G_t$)
Total discounted reward from time step $t$ until the end of the episode:
$$G_t \doteq R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$

### Monte Carlo Evaluation ($Q$)
We estimate the action-value function $q_\pi(s, a)$ by averaging returns across many independent episodes. We track the number of visits $N(s, a)$ to calculate the empirical mean:
$$Q(s, a) = \frac{1}{N(s, a)} \sum_{i=1}^{N(s, a)} G_i(s, a)$$

### Monte Carlo Control ($\pi$)
In Generalized Policy Iteration (GPI), we improve the policy by making it greedy with respect to our current estimates:
$$\pi(s) \doteq \arg\max_a Q(s, a)$$

## 3. Environment Setup: 3x3 GridWorld

<svg width="400" height="400" viewBox="0 0 400 400" xmlns="http://www.w3.org/2000/svg">
  <style>
    .cell { fill: #f8f9fa; stroke: #343a40; stroke-width: 2; }
    .text { font-family: sans-serif; font-size: 16px; fill: #343a40; text-anchor: middle; }
    .reward { font-size: 12px; fill: #6c757d; font-style: italic; }
    .goal { fill: #d4edda; }
    .goal-text { fill: #155724; font-weight: bold; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 1.5; marker-end: url(#arrowhead); opacity: 0.6; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <rect x="50" y="50" width="100" height="100" class="cell" />
  <text x="100" y="80" class="text">State 0</text>
  <text x="100" y="100" class="text reward">R: -1</text>
  <rect x="150" y="50" width="100" height="100" class="cell" />
  <text x="200" y="80" class="text">State 1</text>
  <text x="200" y="100" class="text reward">R: -1</text>
  <rect x="250" y="50" width="100" height="100" class="cell" />
  <text x="300" y="80" class="text">State 2</text>
  <text x="300" y="100" class="text reward">R: -1</text>
  <rect x="50" y="150" width="100" height="100" class="cell" />
  <text x="100" y="180" class="text">State 3</text>
  <text x="100" y="200" class="text reward">R: -1</text>
  <rect x="150" y="150" width="100" height="100" class="cell" />
  <text x="200" y="180" class="text">State 4</text>
  <text x="200" y="200" class="text reward">R: -1</text>
  <rect x="250" y="150" width="100" height="100" class="cell" />
  <text x="300" y="180" class="text">State 5</text>
  <text x="300" y="200" class="text reward">R: -1</text>
  <rect x="50" y="250" width="100" height="100" class="cell" />
  <text x="100" y="280" class="text">State 6</text>
  <text x="100" y="300" class="text reward">R: -1</text>
  <rect x="150" y="250" width="100" height="100" class="cell" />
  <text x="200" y="280" class="text">State 7</text>
  <text x="200" y="300" class="text reward">R: -1</text>
  <rect x="250" y="250" width="100" height="100" class="cell goal" />
  <text x="300" y="280" class="text goal-text">State 8</text>
  <text x="300" y="300" class="text reward goal-text">R: +10</text>
  <path d="M 200 160 L 200 135" class="arrow" />
  <path d="M 200 220 L 200 245" class="arrow" />
  <path d="M 170 190 L 145 190" class="arrow" />
  <path d="M 230 190 L 255 190" class="arrow" />
</svg>

In [ ]:
import numpy as np
import random
from collections import defaultdict

class SimpleGridWorld:
    def __init__(self):
        self.size = 3
        self.goal = 8
    
    def reset(self, start_state=0):
        # For Exploring Starts, we allow forcing the environment to any state
        self.state = start_state
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.size)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(self.size - 1, row + 1) # Down
        elif action == 2: col = max(0, col - 1)  # Left
        elif action == 3: col = min(self.size - 1, col + 1) # Right
        
        self.state = row * self.size + col
        reward = 10 if self.state == self.goal else -1
        done = (self.state == self.goal)
        return self.state, reward, done

def update_q_values(episode, Q, returns_sum, N, gamma=0.9):
    G = 0
    visited_sa = set()
    for t in range(len(episode) - 1, -1, -1):
        s, a, r = episode[t]
        G = gamma * G + r
        # First-visit MC logic
        if (s, a) not in visited_sa:
            visited_sa.add((s, a))
            returns_sum[s][a] += G
            N[s][a] += 1
            Q[s][a] = returns_sum[s][a] / N[s][a]

## 4. Implementation Details

### The "Initial Guess"
We start with an **arbitrary policy** (e.g., "always go Right"). 
- The diversity in our 1000s of episodes comes from either **Exploring Starts** (Method A) or **Epsilon-Greedy Actions** (Method B).

### Why do we need a policy during data collection?
The policy acts as a **guide**. After the first action, the agent follows its plan to ensure it reaches the goal. Without this, the agent might wander forever and never learn anything.

## 5. Method A: Monte Carlo ES (Exploring Starts)

**Logic**: Uses a purely **Deterministic** (Greedy) policy, but forces exploration by starting every episode in a **RANDOM** state and taking a **RANDOM** first action.

In [ ]:
def run_mc_es(num_iterations=20):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))
    policy = defaultdict(lambda: 3) # Initial guess: All Right

    print("--- Running Monte Carlo ES (Random Starts) ---")
    for i in range(num_iterations):
        # EXPLORING START: Force random starting state
        s0 = random.choice(range(8))
        a0 = random.choice(range(4))
        
        episode = []
        state = env.reset(s0)
        
        # Step 1: Execute random first action
        next_state, reward, done = env.step(a0)
        episode.append((state, a0, reward))
        state = next_state
        
        # Step 2: Follow Greedy Policy until terminal
        while not done:
            action = policy[state]
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            
        update_q_values(episode, Q, returns_sum, N)
        
        # Improvement: Make policy greedy w.r.t Q
        for s in range(8):
            policy[s] = np.argmax(Q[s])
            
    print("Final Policy Grid (ES):")
    print(np.array([policy[s] for s in range(9)]).reshape(3,3))

run_mc_es()

## 6. Method B: On-Policy $\epsilon$-Greedy MC

**Logic**: Starts from a **FIXED** state (State 0) every time, but forces exploration by taking **RANDOM ACTIONS** with probability $\epsilon$ throughout the episode.

In [ ]:
def run_mc_epsilon_greedy(num_iterations=200, epsilon=0.2):
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))

    print(f"--- Running Epsilon-Greedy MC (Fixed Start, eps={epsilon}) ---")
    for i in range(num_iterations):
        # FIXED START
        state = env.reset(0) 
        episode = []
        done = False
        
        while not done:
            # EPSILON-GREEDY SELECTION
            if random.random() < epsilon:
                action = random.choice(range(4))
            else:
                action = np.argmax(Q[state])
                
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if len(episode) > 100: break
            
        update_q_values(episode, Q, returns_sum, N)
        
    print("Final Policy Grid (Epsilon-Greedy):")
    final_policy = [np.argmax(Q[s]) for s in range(9)]
    print(np.array(final_policy).reshape(3,3))

run_mc_epsilon_greedy()

## 7. Key Takeaways (On-Policy Methods)

1. **Monte Carlo ES**: Learns fast by sampling the entire world immediately. Best for deterministic policies when you can "reset" the agent anywhere.
2. **$\epsilon$-Greedy**: More realistic. The agent starts at the beginning and must find its own way by occasionally being random. Convergence is slower but more robust for real-world tasks.

Both methods above are **on-policy** — the same policy that generates data is the one being improved. Next, we explore what happens when we separate these two roles.

## 8. Method C: Off-Policy Monte Carlo Control

**Logic**: Uses TWO separate policies:
- **Behavior policy $b$** (fixed, exploratory) — generates episodes by stumbling around randomly
- **Target policy $\pi$** (greedy, improving) — the policy we actually want to learn

The key insight: **b never changes. It just keeps generating random data.** Meanwhile, $\pi$ improves by learning from b's episodes, corrected by importance sampling ratios $\rho = \pi/b$.

<svg width="850" height="400" viewBox="0 0 850 400" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { stroke-width: 2; rx: 8; }
    .title { font-family: sans-serif; font-size: 15px; font-weight: bold; text-anchor: middle; }
    .desc { font-family: sans-serif; font-size: 11px; fill: #495057; text-anchor: middle; }
    .arrow { fill: none; stroke-width: 2.5; marker-end: url(#ah2); }
    .label { font-family: sans-serif; font-size: 11px; font-weight: bold; text-anchor: middle; }
    .section-title { font-family: sans-serif; font-size: 13px; font-weight: bold; }
  </style>
  <defs>
    <marker id="ah2" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#333" />
    </marker>
  </defs>

  <!-- Behavior Policy Box -->
  <rect x="20" y="30" width="200" height="130" class="box" fill="#fff3cd" stroke="#ffc107" />
  <text x="120" y="55" class="title" fill="#856404">Behavior Policy b</text>
  <text x="120" y="80" class="desc">FIXED: Never changes</text>
  <text x="120" y="100" class="desc">Uniform random: b(a|s) = 0.25</text>
  <text x="120" y="120" class="desc">for all 4 actions at every state</text>
  <text x="120" y="145" class="desc" font-style="italic">The "dumb explorer"</text>

  <!-- Arrow: b -> Episodes -->
  <path d="M 220 95 L 270 95" class="arrow" stroke="#ffc107" />
  <text x="245" y="85" class="label" fill="#856404">generates</text>

  <!-- Episodes Box -->
  <rect x="275" y="30" width="200" height="130" class="box" fill="#f8f9fa" stroke="#6c757d" />
  <text x="375" y="55" class="title" fill="#343a40">Episodes</text>
  <text x="375" y="80" class="desc">S₀→a→S₁→a→S₂→...→T</text>
  <text x="375" y="100" class="desc">New random episode each iteration</text>
  <text x="375" y="120" class="desc">Most will hit ρ=0 early</text>
  <text x="375" y="145" class="desc" font-style="italic">(wasted episodes are normal)</text>

  <!-- Arrow: Episodes -> IS Engine -->
  <path d="M 475 95 L 525 95" class="arrow" stroke="#6c757d" />
  <text x="500" y="85" class="label" fill="#6c757d">feed into</text>

  <!-- IS Engine Box -->
  <rect x="530" y="30" width="200" height="130" class="box" fill="#d4edda" stroke="#28a745" />
  <text x="630" y="55" class="title" fill="#155724">Importance Sampling</text>
  <text x="630" y="80" class="desc">For each step: ρ = π(a|s) / b(a|s)</text>
  <text x="630" y="100" class="desc">Weight return: W = W × ρ</text>
  <text x="630" y="120" class="desc">Update: Q ← Q + (W/C)(G - Q)</text>
  <text x="630" y="145" class="desc">If ρ=0: STOP (π rejects action)</text>

  <!-- Arrow: IS -> Target Policy -->
  <path d="M 630 160 L 630 210" class="arrow" stroke="#28a745" />
  <text x="680" y="195" class="label" fill="#155724">improves</text>

  <!-- Target Policy Box -->
  <rect x="530" y="220" width="200" height="120" class="box" fill="#cce5ff" stroke="#007bff" />
  <text x="630" y="250" class="title" fill="#004085">Target Policy π</text>
  <text x="630" y="275" class="desc">GREEDY: π(s) = argmax Q(s,a)</text>
  <text x="630" y="295" class="desc">Fully deterministic</text>
  <text x="630" y="315" class="desc">Improves every iteration</text>
  <text x="630" y="330" class="desc" font-style="italic">The "learner"</text>

  <!-- Arrow: Target Policy -> IS (ratio feedback) -->
  <path d="M 730 270 L 780 270 L 780 95 L 735 95" class="arrow" stroke="#007bff" style="stroke-dasharray: 5,5;" />
  <text x="810" y="185" class="label" fill="#004085" transform="rotate(90,810,185)">ratio π/b</text>

  <!-- Arrow: b loops back to itself -->
  <path d="M 120 160 L 120 200 L 50 200 L 50 60 L 75 60" class="arrow" stroke="#ffc107" />
  <text x="30" y="130" class="label" fill="#856404" transform="rotate(-90,30,130)">loop forever</text>

  <!-- Key difference callout -->
  <rect x="20" y="280" width="460" height="100" fill="#f8d7da" stroke="#dc3545" stroke-width="1.5" rx="8" />
  <text x="250" y="305" class="section-title" fill="#721c24">KEY DIFFERENCE from On-Policy:</text>
  <text x="250" y="330" class="desc" fill="#721c24">On-Policy: π generates episodes AND gets improved (circular)</text>
  <text x="250" y="350" class="desc" fill="#721c24">Off-Policy: b generates episodes (fixed), π gets improved (one-way)</text>
  <text x="250" y="370" class="desc" fill="#721c24">The ρ correction translates b's experience into π's perspective</text>
</svg>

### 8.1 The Two Policy Tables

In off-policy MC, we explicitly maintain two separate policy tables. Here's what they look like for our 3x3 GridWorld:

**Behavior Policy $b$** — Fixed uniform random. Never changes. Guarantees coverage.

| State | $b$(Up) | $b$(Down) | $b$(Left) | $b$(Right) |
|:-----:|:-------:|:---------:|:----------:|:----------:|
| 0 | 0.25 | 0.25 | 0.25 | 0.25 |
| 1 | 0.25 | 0.25 | 0.25 | 0.25 |
| 2 | 0.25 | 0.25 | 0.25 | 0.25 |
| ... | 0.25 | 0.25 | 0.25 | 0.25 |

**Target Policy $\pi$** — Deterministic greedy. Updated every iteration via $\pi(s) = \arg\max_a Q(s,a)$.

| State | $\pi$(Up) | $\pi$(Down) | $\pi$(Left) | $\pi$(Right) |
|:-----:|:---------:|:-----------:|:------------:|:------------:|
| 0 | 0 | 0 | 0 | **1** |
| 1 | 0 | 0 | 0 | **1** |
| 4 | 0 | **1** | 0 | 0 |
| ... | ... | ... | ... | ... |

The importance sampling ratio at each step: $\rho = \frac{\pi(a|s)}{b(a|s)} = \frac{1.0}{0.25} = 4.0$ when $\pi$ agrees, or $\frac{0.0}{0.25} = 0$ when $\pi$ disagrees.

In [ ]:
import numpy as np
import random
from collections import defaultdict

# ============================================================
# 8.2 Define the Two Policies Explicitly
# ============================================================

NUM_STATES = 9
NUM_ACTIONS = 4
ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']

def create_behavior_policy():
    """
    Behavior policy b: Uniform random over all actions.
    Returns a probability TABLE (not a function that picks actions).
    b(a|s) = 0.25 for all s, a. NEVER CHANGES.
    """
    b_table = np.ones((NUM_STATES, NUM_ACTIONS)) / NUM_ACTIONS
    return b_table

def create_target_policy(Q):
    """
    Target policy pi: Deterministic greedy w.r.t. Q.
    Returns a probability TABLE where pi(a|s) = 1 for argmax, 0 otherwise.
    CHANGES EVERY ITERATION as Q improves.
    """
    pi_table = np.zeros((NUM_STATES, NUM_ACTIONS))
    for s in range(NUM_STATES):
        best_action = np.argmax(Q[s])
        pi_table[s, best_action] = 1.0
    return pi_table

# Display the initial policy tables
b_table = create_behavior_policy()
Q_init = np.zeros((NUM_STATES, NUM_ACTIONS))
pi_table = create_target_policy(Q_init)

print("=" * 60)
print("BEHAVIOR POLICY b (fixed, never changes)")
print("=" * 60)
print(f"{'State':<8}", end="")
for a in ACTION_NAMES:
    print(f"{a:<10}", end="")
print()
print("-" * 48)
for s in range(NUM_STATES):
    print(f"  {s:<6}", end="")
    for a in range(NUM_ACTIONS):
        print(f"{b_table[s,a]:<10.2f}", end="")
    print()

print(f"\n{'=' * 60}")
print("TARGET POLICY pi (initially arbitrary - all actions tied)")
print("=" * 60)
print(f"{'State':<8}", end="")
for a in ACTION_NAMES:
    print(f"{a:<10}", end="")
print()
print("-" * 48)
for s in range(NUM_STATES):
    print(f"  {s:<6}", end="")
    for a in range(NUM_ACTIONS):
        marker = " <--" if pi_table[s,a] == 1.0 else ""
        print(f"{pi_table[s,a]:<10.0f}", end="")
    print(f"  (greedy: {ACTION_NAMES[np.argmax(pi_table[s])]})")


### 8.3 Episode Generation from Behavior Policy b

The behavior policy generates episodes by **sampling actions from its probability table**. Since b is uniform random, every action has equal probability (0.25). The episode generator doesn't know or care about the target policy — it just explores randomly.

In [ ]:
def generate_episode_from_b(env, b_table, start_state=0, max_steps=100):
    """
    Generate a single episode using the behavior policy b.
    
    b_table[s] gives the probability distribution over actions at state s.
    We SAMPLE from this distribution — this is how b "explores."
    
    Returns: list of (state, action, reward) tuples
    """
    episode = []
    state = env.reset(start_state)
    done = False
    steps = 0
    
    while not done and steps < max_steps:
        # SAMPLE action from b's probability distribution
        action = np.random.choice(NUM_ACTIONS, p=b_table[state])
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        steps += 1
    
    return episode

# Demonstrate: Generate 5 sample episodes from b
env = SimpleGridWorld()
b_table = create_behavior_policy()

print("=" * 60)
print("SAMPLE EPISODES FROM BEHAVIOR POLICY b (uniform random)")
print("=" * 60)

for ep_num in range(5):
    episode = generate_episode_from_b(env, b_table, start_state=0)
    path = " -> ".join([f"S{s}({ACTION_NAMES[a][0]})" for s, a, r in episode])
    reached_goal = "GOAL!" if episode[-1][2] == 10 else f"(truncated at {len(episode)} steps)"
    print(f"\nEpisode {ep_num+1} ({len(episode)} steps): {reached_goal}")
    # Show first 8 steps
    if len(episode) > 8:
        print(f"  {path[:80]}...")
    else:
        print(f"  {path}")


### 8.4 The Off-Policy MC Control Algorithm (Weighted Importance Sampling)

This is the core algorithm from Sutton & Barto §5.7. The key steps for each episode:

1. **Generate** episode using b (the behavior policy)
2. **Process backwards** from the terminal state
3. At each step, compute $\rho = \frac{\pi(A_t|S_t)}{b(A_t|S_t)}$
4. If $\rho = 0$: **STOP** — $\pi$ would never take this action, so earlier states can't learn from this trajectory
5. If $\rho > 0$: Accumulate the weight $W$, update $Q$ and $C$, improve $\pi$

In [ ]:
def run_mc_off_policy(num_iterations=5000, gamma=0.9, verbose_first_n=5):
    """
    Off-Policy MC Control using Weighted Importance Sampling.
    
    Two policies:
      - b_table: behavior policy (uniform random, FIXED)
      - pi: target policy (greedy w.r.t. Q, IMPROVES each iteration)
    """
    env = SimpleGridWorld()
    
    # Initialize
    Q = np.zeros((NUM_STATES, NUM_ACTIONS))
    C = np.zeros((NUM_STATES, NUM_ACTIONS))  # Cumulative importance sampling weights
    pi = np.zeros(NUM_STATES, dtype=int)      # Target policy: deterministic, stores best action
    b_table = create_behavior_policy()         # Behavior policy: FIXED uniform random
    
    # Statistics tracking
    total_steps = 0             # Total steps across all episodes
    learned_steps = 0           # Steps where Q was actually updated
    fully_used_episodes = 0     # Episodes with NO truncation (every step matched pi)
    
    print("=" * 70)
    print("OFF-POLICY MC CONTROL (Weighted Importance Sampling)")
    print("=" * 70)
    print(f"Behavior policy b: Uniform random (b(a|s) = 0.25 for all s, a)")
    print(f"Target policy pi: Greedy (pi(s) = argmax Q(s,a))")
    print(f"Running {num_iterations} iterations...")
    print("-" * 70)
    
    for i in range(num_iterations):
        # ============================================================
        # STEP 1: Generate episode using behavior policy b
        #         (b never changes — always uniform random)
        # ============================================================
        episode = generate_episode_from_b(env, b_table, start_state=0)
        
        if len(episode) == 0:
            continue
        
        total_steps += len(episode)
            
        # ============================================================
        # STEP 2: Process episode BACKWARDS (from terminal to start)
        # ============================================================
        G = 0.0    # Return accumulator
        W = 1.0    # Importance sampling weight (cumulative product of rho)
        steps_learned_this_episode = 0
        
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            # ============================================================
            # STEP 3: Update Q using weighted importance sampling
            # ============================================================
            C[state, action] += W
            Q[state, action] += (W / C[state, action]) * (G - Q[state, action])
            steps_learned_this_episode += 1
            
            # ============================================================
            # STEP 4: Improve target policy (make it greedy w.r.t. Q)
            # ============================================================
            pi[state] = np.argmax(Q[state])
            
            # ============================================================
            # STEP 5: Compute importance sampling ratio rho
            #         rho = pi(action|state) / b(action|state)
            # ============================================================
            if action != pi[state]:
                # pi would NOT take this action -> rho = 0 / 0.25 = 0
                # STOP: can't learn about earlier states from this trajectory
                break
            
            # pi WOULD take this action -> rho = 1.0 / 0.25 = 4.0
            rho = 1.0 / b_table[state, action]
            W *= rho
        
        # Track statistics
        learned_steps += steps_learned_this_episode
        if steps_learned_this_episode == len(episode):
            fully_used_episodes += 1
        
        # Verbose output for first N episodes
        if i < verbose_first_n:
            path = [f"S{s}({ACTION_NAMES[a][0]})" for s, a, r in episode]
            print(f"\nIteration {i+1}:")
            print(f"  Episode ({len(episode)} steps): {' -> '.join(path[:6])}{'...' if len(episode) > 6 else ''}")
            print(f"  Steps learned from (before rho=0): {steps_learned_this_episode}/{len(episode)}")
            if steps_learned_this_episode < len(episode):
                # Find where truncation happened
                trunc_idx = len(episode) - steps_learned_this_episode
                s_trunc, a_trunc, _ = episode[trunc_idx]
                print(f"  Truncated at step {trunc_idx}: S{s_trunc} took {ACTION_NAMES[a_trunc]}, "
                      f"but pi(S{s_trunc})={ACTION_NAMES[pi[s_trunc]]}")
    
    # ============================================================
    # RESULTS
    # ============================================================
    print(f"\n{'=' * 70}")
    print("RESULTS AFTER TRAINING")
    print(f"{'=' * 70}")
    
    wasted_steps = total_steps - learned_steps
    print(f"\n--- Data Efficiency Statistics ---")
    print(f"  Total episodes generated by b:    {num_iterations}")
    print(f"  Total steps across all episodes:  {total_steps}")
    print(f"  Steps that updated Q (useful):    {learned_steps} ({100*learned_steps/total_steps:.1f}%)")
    print(f"  Steps wasted (rho=0 truncation):  {wasted_steps} ({100*wasted_steps/total_steps:.1f}%)")
    print(f"  Episodes fully used (no trunc.):  {fully_used_episodes} ({100*fully_used_episodes/num_iterations:.1f}%)")
    print(f"\n  --> ~{100*wasted_steps/total_steps:.0f}% of b's random exploration was WASTED")
    print(f"  --> Only {100*fully_used_episodes/num_iterations:.1f}% of episodes contributed to ALL states")
    
    print(f"\n--- Final Target Policy pi (learned) ---")
    policy_grid = np.array([ACTION_NAMES[pi[s]] for s in range(9)]).reshape(3, 3)
    print(policy_grid)
    
    print(f"\n--- Final Q-Table ---")
    print(f"{'State':<8}", end="")
    for a in ACTION_NAMES:
        print(f"{a:<12}", end="")
    print("  Best")
    print("-" * 62)
    for s in range(8):
        print(f"  {s:<6}", end="")
        for a in range(NUM_ACTIONS):
            marker = " *" if a == pi[s] else "  "
            print(f"{Q[s,a]:>8.2f}{marker}", end="")
        print(f"    {ACTION_NAMES[pi[s]]}")
    
    return Q, pi, learned_steps, total_steps

Q_off, pi_off, learned, total = run_mc_off_policy(num_iterations=5000)


### 8.5 Comparing All Three Methods Side-by-Side

Let's run all three methods with the same number of iterations and compare convergence speed, final policies, and data efficiency.

In [ ]:
def run_mc_es_silent(num_iterations=5000, gamma=0.9):
    """Monte Carlo ES - returns Q and policy silently."""
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))
    policy = defaultdict(lambda: 3)

    for i in range(num_iterations):
        s0 = random.choice(range(8))
        a0 = random.choice(range(4))
        episode = []
        state = env.reset(s0)
        next_state, reward, done = env.step(a0)
        episode.append((state, a0, reward))
        state = next_state
        while not done:
            action = policy[state]
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
        update_q_values(episode, Q, returns_sum, N)
        for s in range(8):
            policy[s] = np.argmax(Q[s])
    return Q, policy

def run_mc_onpolicy_silent(num_iterations=5000, epsilon=0.2, gamma=0.9):
    """On-policy epsilon-greedy MC - returns Q and policy silently."""
    env = SimpleGridWorld()
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    N = defaultdict(lambda: np.zeros(4))

    for i in range(num_iterations):
        state = env.reset(0)
        episode = []
        done = False
        while not done:
            if random.random() < epsilon:
                action = random.choice(range(4))
            else:
                action = np.argmax(Q[state])
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if len(episode) > 100: break
        update_q_values(episode, Q, returns_sum, N)
    
    policy = {s: np.argmax(Q[s]) for s in range(9)}
    return Q, policy

def run_mc_offpolicy_stats(num_iterations=5000, gamma=0.9):
    """Off-policy MC - returns Q, policy, and step-level stats."""
    env = SimpleGridWorld()
    Q = np.zeros((NUM_STATES, NUM_ACTIONS))
    C = np.zeros((NUM_STATES, NUM_ACTIONS))
    pi = np.zeros(NUM_STATES, dtype=int)
    b_table = create_behavior_policy()
    total_steps = 0
    learned_steps = 0
    fully_used = 0
    
    for i in range(num_iterations):
        episode = generate_episode_from_b(env, b_table, start_state=0)
        if not episode:
            continue
        total_steps += len(episode)
        G = 0.0
        W = 1.0
        steps_this = 0
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            C[state, action] += W
            Q[state, action] += (W / C[state, action]) * (G - Q[state, action])
            steps_this += 1
            pi[state] = np.argmax(Q[state])
            if action != pi[state]:
                break
            W *= 1.0 / b_table[state, action]
        learned_steps += steps_this
        if steps_this == len(episode):
            fully_used += 1
    
    return Q, pi, total_steps, learned_steps, fully_used

# Run all three methods
print("Running all three methods with 5000 iterations each...\n")
random.seed(42)
np.random.seed(42)

Q_es, pi_es = run_mc_es_silent(5000)
Q_on, pi_on = run_mc_onpolicy_silent(5000)
Q_off, pi_off, total_s, learned_s, fully_used = run_mc_offpolicy_stats(5000)

# Display comparison
print("=" * 70)
print("FINAL POLICY COMPARISON (Action at each grid cell)")
print("=" * 70)
symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}

print("\n  Method A: MC ES          Method B: On-Policy      Method C: Off-Policy")
print("  (Exploring Starts)       (ε-greedy, ε=0.2)       (Uniform b, greedy π)")
print("  " + "-"*20 + "   " + "-"*20 + "   " + "-"*20)

for row in range(3):
    line_es = "  "
    line_on = "  "
    line_off = "  "
    for col in range(3):
        s = row * 3 + col
        if s == 8:
            line_es += " G  "
            line_on += " G  "
            line_off += " G  "
        else:
            line_es += f" {symbols[pi_es[s]]}  "
            line_on += f" {symbols[pi_on[s]]}  "
            line_off += f" {symbols[pi_off[s]]}  "
    print(f"{line_es}      {line_on}      {line_off}")

print(f"\n{'=' * 70}")
print("DATA EFFICIENCY COMPARISON")
print(f"{'=' * 70}")
print(f"\n  {'Method':<25} {'Steps Generated':<18} {'Steps Used':<18} {'Utilization'}")
print(f"  {'-'*25} {'-'*18} {'-'*18} {'-'*12}")
print(f"  {'MC ES (Method A)':<25} {'~all short':<18} {'~all':<18} {'~100%'}")
print(f"  {'On-Policy (Method B)':<25} {'~all':<18} {'~all':<18} {'~100%'}")
print(f"  {'Off-Policy (Method C)':<25} {total_s:<18} {learned_s:<18} {100*learned_s/total_s:.1f}%")
print(f"\n  Off-policy episodes fully used (no rho=0 truncation): "
      f"{fully_used}/5000 = {100*fully_used/5000:.1f}%")
print(f"\n  KEY TAKEAWAY: Off-policy wastes ~{100*(total_s-learned_s)/total_s:.0f}% of steps,")
print(f"  but it can learn the TRUE optimal policy (no epsilon compromise).")


### 8.6 Visualizing the Difference: What Each Method "Sees"

The following diagram shows how data flows differently in each method, and why off-policy wastes episodes:

<svg width="850" height="520" viewBox="0 0 850 520" xmlns="http://www.w3.org/2000/svg">
  <style>
    .method-title { font-family: sans-serif; font-size: 14px; font-weight: bold; text-anchor: middle; }
    .step-text { font-family: monospace; font-size: 11px; fill: #343a40; }
    .good { fill: #d4edda; stroke: #28a745; }
    .bad { fill: #f8d7da; stroke: #dc3545; }
    .neutral { fill: #f8f9fa; stroke: #6c757d; }
    .arrow-good { fill: none; stroke: #28a745; stroke-width: 2; marker-end: url(#ah3); }
    .arrow-bad { fill: none; stroke: #dc3545; stroke-width: 2; marker-end: url(#ah4); }
  </style>
  <defs>
    <marker id="ah3" markerWidth="8" markerHeight="6" refX="0" refY="3" orient="auto">
      <polygon points="0 0, 8 3, 0 6" fill="#28a745" />
    </marker>
    <marker id="ah4" markerWidth="8" markerHeight="6" refX="0" refY="3" orient="auto">
      <polygon points="0 0, 8 3, 0 6" fill="#dc3545" />
    </marker>
  </defs>

  <!-- Method A: MC ES -->
  <text x="140" y="25" class="method-title" fill="#007bff">Method A: MC ES</text>
  <rect x="20" y="35" width="240" height="145" class="good" stroke-width="1.5" rx="6" />
  <text x="30" y="55" class="step-text">Episode: S4(↓)→S7(→)→S8 ✓</text>
  <text x="30" y="75" class="step-text">  Random start: S4</text>
  <text x="30" y="95" class="step-text">  Random 1st action: Down</text>
  <text x="30" y="115" class="step-text">  Then follow greedy policy</text>
  <text x="30" y="140" class="step-text" style="fill:#155724; font-weight:bold">  ALL steps update Q ✓</text>
  <text x="30" y="160" class="step-text" style="fill:#155724">  No wasted data</text>

  <!-- Method B: On-Policy -->
  <text x="430" y="25" class="method-title" fill="#28a745">Method B: On-Policy ε-greedy</text>
  <rect x="305" y="35" width="250" height="145" class="good" stroke-width="1.5" rx="6" />
  <text x="315" y="55" class="step-text">Episode: S0(→)→S1(→)→S2(↓)→S5(↓)→S8 ✓</text>
  <text x="315" y="75" class="step-text">  Fixed start: S0</text>
  <text x="315" y="95" class="step-text">  Each step: 80% greedy, 20% random</text>
  <text x="315" y="115" class="step-text">  Same policy generates AND learns</text>
  <text x="315" y="140" class="step-text" style="fill:#155724; font-weight:bold">  ALL steps update Q ✓</text>
  <text x="315" y="160" class="step-text" style="fill:#155724">  No correction needed (data matches policy)</text>

  <!-- Method C: Off-Policy -->
  <text x="710" y="25" class="method-title" fill="#dc3545">Method C: Off-Policy</text>
  <rect x="580" y="35" width="260" height="145" class="bad" stroke-width="1.5" rx="6" />
  <text x="590" y="55" class="step-text">Episode: S0(→)→S1(↑)→S1(←)→S0(→)→...</text>
  <text x="590" y="75" class="step-text">  Fixed start: S0</text>
  <text x="590" y="95" class="step-text">  b picks uniformly random each step</text>
  <text x="590" y="115" class="step-text">  Process backwards: last step ρ=4 ✓</text>
  <text x="590" y="135" class="step-text" style="fill:#721c24">  Hit step where b took ↑ but π says →</text>
  <text x="590" y="155" class="step-text" style="fill:#721c24; font-weight:bold">  ρ=0: STOP. Earlier steps WASTED ✗</text>
  <text x="590" y="170" class="step-text" style="fill:#721c24">  Only tail of episode is useful</text>

  <!-- Efficiency comparison bars -->
  <text x="425" y="215" class="method-title" fill="#343a40">Data Efficiency Comparison (same 5000 episodes)</text>
  
  <!-- ES bar -->
  <rect x="50" y="235" width="300" height="30" fill="#28a745" rx="4" />
  <text x="60" y="255" class="step-text" style="fill:white; font-weight:bold">MC ES: ~100% data utilization</text>

  <!-- On-Policy bar -->
  <rect x="50" y="275" width="300" height="30" fill="#28a745" rx="4" />
  <text x="60" y="295" class="step-text" style="fill:white; font-weight:bold">On-Policy: ~100% data utilization</text>

  <!-- Off-Policy bar (partially red) -->
  <rect x="50" y="315" width="300" height="30" fill="#dc3545" rx="4" />
  <rect x="50" y="315" width="195" height="30" fill="#ffc107" rx="4" />
  <text x="60" y="335" class="step-text" style="fill:#343a40; font-weight:bold">Off-Policy: ~65% useful | 35% wasted (ρ=0)</text>

  <!-- Trade-off summary -->
  <rect x="50" y="370" width="750" height="140" fill="#e2e3e5" stroke="#6c757d" stroke-width="1" rx="8" />
  <text x="425" y="395" class="method-title" fill="#343a40">WHY USE OFF-POLICY THEN?</text>
  
  <text x="70" y="420" class="step-text" style="fill:#155724">✓ Can learn the OPTIMAL (fully greedy) policy — no ε needed</text>
  <text x="70" y="440" class="step-text" style="fill:#155724">✓ Can reuse OLD data from ANY source (logs, humans, other agents)</text>
  <text x="70" y="460" class="step-text" style="fill:#155724">✓ Separates exploration (b's job) from exploitation (π's job)</text>
  <text x="70" y="485" class="step-text" style="fill:#721c24">✗ Wastes many episodes (ρ=0 truncation)</text>
  <text x="70" y="505" class="step-text" style="fill:#721c24">✗ High variance (ρ products grow exponentially with episode length)</text>
</svg>

### 8.7 Advantages and Disadvantages Summary

| Aspect | MC ES (Method A) | On-Policy ε-greedy (Method B) | Off-Policy (Method C) |
|:-------|:-----------------|:------------------------------|:----------------------|
| **Exploration mechanism** | Random starting state + action | ε-random actions during episode | Separate behavior policy b |
| **Policy type learned** | Deterministic greedy | ε-soft (suboptimal) | Deterministic greedy (optimal) |
| **Data efficiency** | High (all data useful) | High (all data useful) | Low (many episodes truncated by ρ=0) |
| **Requires reset to any state?** | YES (impractical for real robots) | NO | NO |
| **Can reuse old data?** | NO | NO (old data is from old π) | YES (any data from any b works) |
| **Convergence speed** | Fast | Medium | Slow (needs many episodes) |
| **Variance** | Low | Low | High (ρ products explode) |
| **Real-world applicability** | Low (need full reset control) | High | High (can learn from logs/humans) |
| **Can learn truly optimal π?** | YES | NO (ε forces suboptimality) | YES |

### Key Insight: The Fundamental Trade-Off

- **On-policy** methods are simple and stable, but can never learn the truly optimal policy (they must keep exploring with ε > 0).
- **Off-policy** methods can learn the optimal policy, but pay for it with high variance and wasted data.
- This trade-off is what motivated **Q-learning** (Chapter 6): it achieves off-policy learning WITHOUT importance sampling, by bootstrapping one step at a time instead of waiting for full episodes.